# Daily Challenge – Week 5, Day 3: Bank Customer Churn Prediction
Predict which customers are likely to **leave the bank** (`Exited` = 1), so the bank can focus retention efforts on them.
Dataset: `Churn_Modelling.csv` (10,000 customers).

## 1. Load and Explore the Data

In [ ]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score, roc_auc_score,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report, RocCurveDisplay)

LOCAL_FILE = "Churn_Modelling.csv"
GITHUB_URL = "https://raw.githubusercontent.com/jenny-palti/DI-Bootcamp2026/main/Week5/Day3/Churn_Modelling.csv"
df = pd.read_csv(LOCAL_FILE if os.path.exists(LOCAL_FILE) else GITHUB_URL)
print(df.shape)
df.head()

In [ ]:
df.info()
print("\nMissing values:", df.isnull().sum().sum())
print("Duplicate rows:", df.duplicated().sum())
print(f"\nChurn rate: {df['Exited'].mean():.1%}")

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9))
sns.countplot(data=df, x="Exited", hue="Exited", palette="Set2", legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Churn (1 = left the bank)")

for ax, col in zip([axes[0, 1], axes[0, 2], axes[1, 0]], ["Geography", "NumOfProducts", "IsActiveMember"]):
    df.groupby(col)["Exited"].mean().plot(kind="bar", ax=ax, color="indianred")
    ax.set_title(f"Churn rate by {col}")
    ax.set_ylabel("Churn rate")
    ax.tick_params(axis="x", rotation=0)

sns.boxplot(data=df, x="Exited", y="Age", ax=axes[1, 1])
axes[1, 1].set_title("Age by churn")
sns.boxplot(data=df, x="Exited", y="Balance", ax=axes[1, 2])
axes[1, 2].set_title("Balance by churn")
plt.tight_layout()
plt.show()

In [ ]:
for col in ["Geography", "Gender", "NumOfProducts", "IsActiveMember", "HasCrCard", "Tenure"]:
    print(df.groupby(col)["Exited"].agg(churn_rate="mean", customers="count").round(3), "\n")

## 2. Data Preprocessing
- **Drop** `RowNumber`, `CustomerId` and `Surname`: identifiers that carry no information about behaviour (and using a surname could introduce bias).
- **Missing values:** none in this file, but the pipeline includes imputers (median for numbers, most frequent for categories) so it would still work on new data with gaps.
- **Encode** `Geography` and `Gender` with one-hot encoding.
- **Scale** numerical features (needed for logistic regression).
- **Stratified** train/test split, because only ~20% of customers churn.

In [ ]:
df_model = df.drop(columns=["RowNumber", "CustomerId", "Surname"])
X = df_model.drop(columns="Exited")
y = df_model["Exited"]

categorical = ["Geography", "Gender"]
numerical = [c for c in X.columns if c not in categorical]

preprocess = ColumnTransformer([
    ("num", Pipeline([("impute", SimpleImputer(strategy="median")), ("scale", StandardScaler())]), numerical),
    ("cat", Pipeline([("impute", SimpleImputer(strategy="most_frequent")),
                      ("onehot", OneHotEncoder(drop="first", handle_unknown="ignore"))]), categorical),
])

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)
print("Train:", X_train.shape, "| Test:", X_test.shape, "| churn rate in test:", round(y_test.mean(), 3))

## 3. Build and Evaluate the Models
A simple, explainable **Logistic Regression** and a more flexible **Random Forest**. `class_weight="balanced"` makes the models pay more attention to the smaller churn class.

In [ ]:
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, class_weight="balanced"),
    "Random Forest": RandomForestClassifier(n_estimators=300, max_depth=10, min_samples_leaf=5,
                                            class_weight="balanced", random_state=42, n_jobs=-1),
}

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
fitted, rows = {}, []
for name, model in models.items():
    pipe = Pipeline([("prep", preprocess), ("model", model)])
    cv_auc = cross_val_score(pipe, X_train, y_train, cv=cv, scoring="roc_auc").mean()
    pipe.fit(X_train, y_train)
    pred = pipe.predict(X_test)
    proba = pipe.predict_proba(X_test)[:, 1]
    fitted[name] = pipe
    rows.append({"model": name, "CV ROC-AUC": cv_auc, "accuracy": accuracy_score(y_test, pred),
                 "precision": precision_score(y_test, pred), "recall": recall_score(y_test, pred),
                 "F1": f1_score(y_test, pred), "test ROC-AUC": roc_auc_score(y_test, proba)})

results = pd.DataFrame(rows).set_index("model").round(3)
results

In [ ]:
best_name = results["test ROC-AUC"].idxmax()
best = fitted[best_name]
print("Best model:", best_name)
print(classification_report(y_test, best.predict(X_test), target_names=["Stayed", "Churned"], digits=3))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
ConfusionMatrixDisplay.from_estimator(best, X_test, y_test, display_labels=["Stayed", "Churned"], cmap="Blues", ax=axes[0])
axes[0].set_title(f"Confusion matrix – {best_name}")
for name, pipe in fitted.items():
    RocCurveDisplay.from_estimator(pipe, X_test, y_test, name=name, ax=axes[1])
axes[1].plot([0, 1], [0, 1], "k--")
axes[1].set_title("ROC curves")
plt.tight_layout()
plt.show()

## 4. What Drives Churn?

In [ ]:
rf_pipe = fitted["Random Forest"]
feature_names = rf_pipe.named_steps["prep"].get_feature_names_out()
importances = pd.Series(rf_pipe.named_steps["model"].feature_importances_, index=feature_names)
importances.index = importances.index.str.replace("num__", "").str.replace("cat__", "")

importances.sort_values().plot(kind="barh", figsize=(8, 5), color="steelblue")
plt.title("Random Forest – feature importance")
plt.show()

lr_pipe = fitted["Logistic Regression"]
coefs = pd.Series(lr_pipe.named_steps["model"].coef_[0], index=importances.index)
print("Logistic regression coefficients (positive = higher churn risk):")
coefs.sort_values(ascending=False).round(3)

In [ ]:
# Practical use: rank customers by predicted churn risk
risk = X_test.copy()
risk["churn_probability"] = best.predict_proba(X_test)[:, 1]
risk["actually_churned"] = y_test
top = risk.sort_values("churn_probability", ascending=False)

top_10pct = top.head(len(top) // 10)
print(f"Churn rate among the 10% highest-risk customers: {top_10pct['actually_churned'].mean():.1%}"
      f" (vs. {y_test.mean():.1%} overall)")
top.head(10)[["Age", "Geography", "Balance", "NumOfProducts", "IsActiveMember", "churn_probability", "actually_churned"]]

## 5. Insights and Business Recommendations
**Model results**
- The **Random Forest** is clearly better: test ROC-AUC **0.87** vs. 0.78 for logistic regression (cross-validation shows the same: 0.86 vs. 0.77). Churn depends on non-linear patterns, which a straight-line model can't capture. For example, risk is low with 2 products but very high with 3–4.
- With balanced class weights, the Random Forest catches **74% of the customers who actually leave** (recall), with precision 54%: about half of the customers it flags really churn. For a retention campaign, that trade-off is usually worth it, since contacting a loyal customer costs little.
- **Ranking works very well:** among the **10% of customers with the highest predicted risk, 83% actually churned**, compared with 20% overall. The bank can focus its efforts on that list.

**What drives churn**
1. **Age** is the strongest factor. Customers who left have a median age of **45 vs. 36** for those who stayed; middle-aged and older customers leave most often.
2. **Number of products:** customers with **2 products churn the least (8%)**. With only 1 product churn is 28%, and almost everyone with 3–4 products left (83–100%), which suggests those customers were sold products that didn't suit them.
3. **Activity:** **inactive members churn about twice as often** (27% vs. 14%).
4. **Geography:** customers in **Germany churn twice as much** (32%) as in France or Spain (~16–17%).
5. **Balance:** churners have *higher* median balances (~109k vs. ~92k), so the bank is losing valuable customers.
6. **Gender:** women churned more often than men (25% vs. 17%) in this data.
7. Credit card ownership, tenure and estimated salary have almost no effect.

**Business recommendations**
- **Run a monthly churn score** and contact the top-risk customers (e.g. top 10%) with personal offers or a call from an advisor.
- **Re-engage inactive customers** (app notifications, rewards for using the account), since activity is one of the few drivers the bank can directly influence.
- **Review the 3–4 product bundles:** the near-100% churn suggests over-selling or poor fit. Aim to move single-product customers to a well-matched second product instead.
- **Investigate Germany:** a churn rate twice as high points to a local issue (competition, fees, service quality) worth researching.
- **Create retention programmes for customers aged ~40–60 with high balances.** They are both the most likely to leave and the most valuable.
- **Fairness note:** gender and age are informative, but the bank should use them to understand and *help* customers, not to treat them unfairly. Any automated decisions should be checked for bias.